# KionTTS Kaggle Training & Validation Pipeline (2x T4 GPUs)

This notebook executes the end-to-end training and validation for **KionTTS**:
- **Architecture**: StyleTTS2 with dedicated **KionTagStyleEncoder** (24 emotion/style tags to 256-d latent style vector).
- **Phonemizer**: Strict official 178-symbol dictionary from text_utils.py (prevents token shift/scrambled syllables).
- **Fast Test Configuration**:
  - Sample dataset: 500 train samples + 120 val samples.
  - **Total Epochs**: 6 epochs
  - **Stage 1**: Epochs 0 to 2 (Predictor & Style Alignment with intact decoder).
  - **Stage 2**: Epochs 3 to 5 (Joint training transition with verified unfreezing & true backpropagated gradients).
  - Audio evaluation generated after every epoch.


In [ ]:
# Cell 01: Verify GPU Environment & Install System Dependencies
import os
import sys
import torch

print(f'[*] PyTorch Version: {torch.__version__}')
print(f'[*] CUDA Available: {torch.cuda.is_available()}')
print(f'[*] GPU Count: {torch.cuda.device_count()}')
for i in range(torch.cuda.device_count()):
    print(f'    GPU {i}: {torch.cuda.get_device_name(i)} ({torch.cuda.get_device_properties(i).total_memory / 1e9:.2f} GB)')

# Install system espeak-ng and python packages
!apt-get update -qq && apt-get install -y -qq espeak-ng > /dev/null
!pip install -q soundfile torchaudio munch librosa phonemizer einops transformers
print('[✓] Dependencies installed successfully.')


In [ ]:
# Cell 02: Setup Workspace & Download Pretrained Weights
import os
import urllib.request

WORKSPACE_DIR = '/kaggle/working'
os.chdir(WORKSPACE_DIR)

os.makedirs('/kaggle/working/Models/LibriTTS', exist_ok=True)
os.makedirs('/kaggle/working/Utils/ASR', exist_ok=True)
os.makedirs('/kaggle/working/Utils/JDC', exist_ok=True)
os.makedirs('/kaggle/working/Utils/PLBERT', exist_ok=True)

libri_ckpt = '/kaggle/working/Models/LibriTTS/epochs_2nd_00020.pth'
if not os.path.exists(libri_ckpt):
    print('[*] Downloading pretrained LibriTTS checkpoint (epochs_2nd_00020.pth)...')
    url = 'https://huggingface.co/yl4579/StyleTTS2-LibriTTS/resolve/main/epochs_2nd_00020.pth'
    urllib.request.urlretrieve(url, libri_ckpt)
    print(f'[✓] Downloaded LibriTTS checkpoint ({os.path.getsize(libri_ckpt) / 1024 / 1024:.2f} MB).')
else:
    print(f'[✓] Found LibriTTS checkpoint at {libri_ckpt}')


In [ ]:
# Cell 03: Prepare Sample Dataset (Auto-Detects Auto-Extracted Kaggle Input) & Pre-phonemize
import os
import shutil
import json
import tarfile
import phonemizer

def locate_dataset():
    # 1. Search in /kaggle/input for auto-extracted datasets
    if os.path.exists('/kaggle/input'):
        for root, dirs, files in os.walk('/kaggle/input'):
            if 'train_manifest.json' in files:
                print(f'[✓] Detected auto-extracted Kaggle dataset at: {root}')
                return root, False
            for f in files:
                if f.endswith('.tar.gz') or f.endswith('.tar') or f.endswith('.zip'):
                    return os.path.join(root, f), True
    # 2. Local fallback
    local_candidates = ['DataSet/sample_kion', 'dataset/sample_kion']
    for c in local_candidates:
        if os.path.exists(os.path.join(c, 'train_manifest.json')):
            return os.path.abspath(c), False
    if os.path.exists('DataSet/sample_kion.tar.gz'):
        return os.path.abspath('DataSet/sample_kion.tar.gz'), True
    raise FileNotFoundError('Could not find sample dataset in /kaggle/input or local repository!')

source_path, is_archive = locate_dataset()

# Writable working directory for manifests
working_data_dir = '/kaggle/working/dataset/sample_kion'
os.makedirs(working_data_dir, exist_ok=True)

if is_archive:
    print(f'[*] Extracting archive {source_path} to {working_data_dir}...')
    with tarfile.open(source_path, 'r:*') as tar:
        tar.extractall('/kaggle/working/dataset')
    audio_root_dir = working_data_dir
else:
    # Auto-extracted in /kaggle/input (read-only audio)
    audio_root_dir = source_path
    for fname in ['train_manifest.json', 'val_manifest.json', 'tag_mapping.json']:
        src_f = os.path.join(source_path, fname)
        if os.path.exists(src_f):
            shutil.copy(src_f, os.path.join(working_data_dir, fname))

print(f'[✓] Audio root directory: {audio_root_dir}')
print(f'[✓] Writable manifest directory: {working_data_dir}')

# Pre-phonemize manifests using official EspeakBackend
global_phonemizer = phonemizer.backend.EspeakBackend(
    language='en-us', preserve_punctuation=True, with_stress=True
)

for split in ['train', 'val']:
    manifest_path = os.path.join(working_data_dir, f'{split}_manifest.json')
    with open(manifest_path, 'r', encoding='utf-8') as f:
        items = json.load(f)
    
    texts_to_phonemize = [it['clean_text'] for it in items]
    print(f'[*] Phonemizing {len(texts_to_phonemize)} {split} samples...')
    phonemized_texts = global_phonemizer.phonemize(texts_to_phonemize)
    for it, ps in zip(items, phonemized_texts):
        it['phonemes'] = ps.strip()
        
    with open(manifest_path, 'w', encoding='utf-8') as f:
        json.dump(items, f, indent=2)
        
print('[✓] Pre-phonemization complete with official 178-token IPA representation.')


In [ ]:
# Cell 04: Initialize KionTTS Model & KionTagStyleEncoder
import yaml
from munch import Munch
import torch

from kion_core.tag_style_encoder import KionTagStyleEncoder
from kion_core.dataset import build_kion_dataloader
from StyleTTS2.models import build_model
from StyleTTS2.Utils.ASR.models import ASRCNN
from StyleTTS2.Utils.JDC.model import JDCNet
from StyleTTS2.Utils.PLBERT.util import load_plbert

device = 'cuda' if torch.cuda.is_available() else 'cpu'

config_path = 'StyleTTS2/Configs/config_ft.yml'
with open(config_path) as f:
    config = yaml.safe_load(f)

model_params = Munch(config['model_params'])
model_params.multispeaker = False  # Single speaker Kion

print('[*] Loading auxiliary ASR, JDC, and PLBERT models...')
text_aligner = ASRCNN(64, 512, 178)
text_aligner.load_state_dict(torch.load('StyleTTS2/Utils/ASR/epoch_00080.pth', map_location='cpu')['model'])
text_aligner = text_aligner.to(device).eval()

pitch_extractor = JDCNet(num_class=1, seq_len=192)
pitch_extractor.load_state_dict(torch.load('StyleTTS2/Utils/JDC/bst.t7', map_location='cpu')['net'])
pitch_extractor = pitch_extractor.to(device).eval()

bert = load_plbert('StyleTTS2/Utils/PLBERT/')
bert = bert.to(device).eval()

print('[*] Building StyleTTS2 model and loading LibriTTS pretrained weights...')
model = build_model(model_params, text_aligner, pitch_extractor, bert)
pretrained_dict = torch.load('/kaggle/working/Models/LibriTTS/epochs_2nd_00020.pth', map_location='cpu')['net']

for key in model.keys():
    if key in pretrained_dict:
        try:
            model[key].load_state_dict(pretrained_dict[key], strict=False)
        except Exception as e:
            print(f'    Warning loading {key}: {e}')
    model[key] = model[key].to(device)

print('[*] Initializing KionTagStyleEncoder...')
tag_encoder = KionTagStyleEncoder(num_tags=24, emb_dim=64, style_dim=256).to(device)
print('[✓] Model architecture initialized successfully.')


In [ ]:
# Cell 05: Execute 2-Stage Fast Test Training
from kion_core.trainer import run_training_pipeline

train_loader = build_kion_dataloader(
    manifest_path=os.path.join(working_data_dir, 'train_manifest.json'),
    root_dir=audio_root_dir,
    batch_size=4,
    validation=False,
    num_workers=2
)
val_loader = build_kion_dataloader(
    manifest_path=os.path.join(working_data_dir, 'val_manifest.json'),
    root_dir=audio_root_dir,
    batch_size=4,
    validation=True,
    num_workers=2
)

train_config = {
    'epochs': 6,          # 6 epochs total for smoke test
    'joint_epoch': 3,     # Transition to Stage 2 at Epoch 3
    'lr': 1e-4,
    'accum_steps': 2,     # Effective batch size 8, safe for 16GB T4
    'save_freq': 1
}

final_ckpt_path = run_training_pipeline(
    model=model,
    tag_encoder=tag_encoder,
    train_loader=train_loader,
    val_loader=val_loader,
    config=train_config,
    device=device,
    log_dir='/kaggle/working/checkpoints',
    eval_dir='/kaggle/working/eval_samples',
    phonemizer_fn=global_phonemizer.phonemize
)


In [ ]:
# Cell 06: Post-Training Inference Sanity Check
import soundfile as sf
import IPython.display as ipd
from kion_core.synthesizer import KionSynthesizer

synthesizer = KionSynthesizer(model, tag_encoder, phonemizer_fn=global_phonemizer.phonemize, device=device)

test_sentences = [
    '[happy=0.9] I finally finished building KionTTS, and it works wonderfully!',
    '[sarcasm=0.8] Oh, fantastic. Another bug to investigate.',
    '[calm=0.8, soothing=0.7] Don't worry at all. We have full control over the speaking style now.'
]

print('[*] Synthesizing test prompts...')
for i, prompt in enumerate(test_sentences):
    wave = synthesizer.synthesize(prompt=prompt)
    print(f'\nPrompt {i+1}: {prompt}')
    out_wav = f'/kaggle/working/eval_samples/final_test_{i+1}.wav'
    sf.write(out_wav, wave, 24000)
    ipd.display(ipd.Audio(wave, rate=24000))


In [ ]:
# Cell 07: Export Final KionTTS Release Bundle
from kion_core.export import export_kion_model

tag_mapping_path = os.path.join(working_data_dir, 'tag_mapping.json')
bundle_path = export_kion_model(
    model=model,
    tag_encoder=tag_encoder,
    config=train_config,
    tag_mapping_path=tag_mapping_path,
    output_dir='/kaggle/working/kiontts_release'
)
print('[✓] Pipeline complete! Release bundle ready at /kaggle/working/kiontts_release.')
